# Сверка клиентов ТЭ: выгрузка Снигура (ЦФТ) и моя выгрузка

Моя выгрузка - лист `С ТЭ на 01.09.2026` из `trade_acquiring_clients_month_excel.ipynb` (файл `trade_acquiring_clients_2026-09-01.xlsx`).

Выгрузка Снигура - Excel из ЦФТ: наименование клиента, номер договора, дата начала, дата окончания, дата закрытия, статус (Закрыт / Работает / На закрытие). ИНН нет, поэтому ключ - **номер договора** (без пробелов, в верхнем регистре).

Какие договоры Снигура идут в сравнение, задаёт `SNIGUR_ACTIVE_RULE`:
- `'all'` (по умолчанию) - весь файл, как он есть: Снигур считает актуальными все договоры в нём;

Для `'end'` и `'close'` договор действует, если начало не позже 01.09.2026, а конец пустой или не раньше 01.09.2026:
- `'end'` - конец = более ранняя из «Даты окончания действия договора» и «Даты закрытия». Так же, как срок действия в `ods_alpha` и `final_df`: договор «На закрытии» с истёкшим сроком не считается действующим;
- `'close'` - «Дата закрытия», а если она пустая - дата окончания. Так договор считается клиентом до формального закрытия.

Статус выводится для контроля.

Для расхождений причина берётся из `ods_alpha.scd1_agreements` + `scd1_agr_terms` - те же фильтры, что в моей выгрузке (SA, действует на дату, привязка `P`).

Положить файл Снигура в `/home/jovyan/documents/Equaring` и указать имя в `SNIGUR_FILE`. Результат - `te_snigur_vs_mine_2026-09-01.xlsx` там же.

Для сравнения «до и после правок озера» старую выгрузку переименовать в `trade_acquiring_clients_2026-09-01_before_lake_fix.xlsx` (`BEFORE_FILE`) до пересборки. Если файла нет, этот блок пропускается.


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

AS_OF_DATE = '2026-09-01'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
SNIGUR_FILE = BASE_DIR / 'Скрипт для Ани по сборке продукта эквайринг' / 'cdwh_acq_clients.xlsx'
SNIGUR_SHEET = 0
SNIGUR_ACTIVE_RULE = 'all'  # 'all' - весь файл, как считает Снигур; 'end' - по более ранней из дат окончания и закрытия; 'close' - по дате закрытия
MINE_FILE = BASE_DIR / ('trade_acquiring_clients_%s.xlsx' % AS_OF_DATE)
OUT_FILE = BASE_DIR / ('te_snigur_vs_mine_%s.xlsx' % AS_OF_DATE)
BEFORE_FILE = BASE_DIR / ('trade_acquiring_clients_%s_before_lake_fix.xlsx' % AS_OF_DATE)

as_of = pd.Timestamp(AS_OF_DATE)
as_of_sql = as_of.strftime('%Y-%m-%d')
as_of_ru = as_of.strftime('%d.%m.%Y')
MINE_SHEET = 'С ТЭ на %s' % as_of_ru
print(SNIGUR_FILE, MINE_FILE, MINE_SHEET, sep='\n')

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


## 1. Загрузка файлов

In [ ]:
def norm_agr(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0', s):
        s = s[:-2]
    s = re.sub(r'\s+', '', s).upper()
    if s in ('', 'NONE', 'NAN', '<NA>'):
        return None
    return s

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    return out.dt.normalize()

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def find_col(df, *keys, exact=None, required=True):
    if exact and exact in df.columns:
        return exact
    hits = [c for c in df.columns if all(k in _low(c) for k in keys)]
    if len(hits) == 1:
        return hits[0]
    if not hits and not required:
        return None
    raise KeyError('Колонка %s: найдено %s. Все колонки: %s' % (keys, hits, list(df.columns)))

def read_with_header(path, sheet, marker='договор', max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        if any(marker in _low(v) for v in raw.iloc[i] if pd.notna(v)):
            df = raw.iloc[i + 1:].copy()
            df.columns = [_low(v) if pd.notna(v) else 'col_%s' % j for j, v in enumerate(raw.iloc[i])]
            return df.dropna(how='all').reset_index(drop=True)
    raise ValueError('Не нашёл строку заголовков с "%s" в первых %s строках' % (marker, max_rows))

sn_raw = read_with_header(SNIGUR_FILE, SNIGUR_SHEET)
c_name = find_col(sn_raw, 'наимен', exact='наименование клиента')
c_agr = find_col(sn_raw, 'номер', 'договор', exact='номер договора')
c_start = find_col(sn_raw, 'начал', exact='дата начала действия договора')
c_end = find_col(sn_raw, 'окончан', exact='дата окончания действия договора')
c_close = find_col(sn_raw, 'закрыт', 'дата', exact='дата закрытия')
c_status = find_col(sn_raw, 'статус', exact='статус')
print('Колонки Снигура:', c_name, c_agr, c_start, c_end, c_close, c_status, sep=' | ')

snigur = pd.DataFrame({
    'agr_key': sn_raw[c_agr].map(norm_agr),
    'Клиент (Снигур)': sn_raw[c_name].astype('string').str.strip(),
    'Номер договора (Снигур)': sn_raw[c_agr].astype('string').str.strip(),
    'Дата начала (Снигур)': to_date(sn_raw[c_start]),
    'Дата окончания (Снигур)': to_date(sn_raw[c_end]),
    'Дата закрытия (Снигур)': to_date(sn_raw[c_close]),
    'Статус (Снигур)': sn_raw[c_status].astype('string').str.strip(),
})
print('Строк у Снигура:', len(snigur), '| без номера договора:', int(snigur['agr_key'].isna().sum()))
snigur = snigur[snigur['agr_key'].notna()].reset_index(drop=True)

mine_raw = pd.read_excel(MINE_FILE, sheet_name=MINE_SHEET, dtype=object)
mine = pd.DataFrame({
    'agr_key': mine_raw['Номер договора'].map(norm_agr),
    'ИНН': mine_raw['ИНН'].astype('string'),
    'ЦФТ ID': mine_raw['ЦФТ ID'].astype('string'),
    'Номер договора (мой)': mine_raw['Номер договора'].astype('string'),
    'Дата открытия (мой)': to_date(mine_raw['Дата открытия договора']),
    'Дата закрытия (мой)': to_date(mine_raw['Дата закрытия договора']),
    'РФ': mine_raw['РФ, в котором обслуживается клиент'],
})
mine = mine[mine['agr_key'].notna()]
mine_rows = mine.groupby('agr_key').size().rename('Строк у меня')
mine = mine.drop_duplicates('agr_key').merge(mine_rows, on='agr_key').reset_index(drop=True)
print('Моих договоров:', len(mine), '| номеров с повтором:', int((mine['Строк у меня'] > 1).sum()))
display(snigur.head(10))
display(mine.head(10))


## 2. Действующие у Снигура на дату

In [ ]:
if SNIGUR_ACTIVE_RULE == 'end':
    sn_end = snigur[['Дата окончания (Снигур)', 'Дата закрытия (Снигур)']].min(axis=1)
elif SNIGUR_ACTIVE_RULE in ('close', 'all'):
    sn_end = snigur['Дата закрытия (Снигур)'].fillna(snigur['Дата окончания (Снигур)'])
else:
    raise ValueError("SNIGUR_ACTIVE_RULE: 'all', 'end' или 'close'")
print('Правило «действует у Снигура»:', SNIGUR_ACTIVE_RULE)
snigur['Конец (Снигур)'] = sn_end
if SNIGUR_ACTIVE_RULE == 'all':
    snigur['Действует на дату'] = True
else:
    snigur['Действует на дату'] = (
        (snigur['Дата начала (Снигур)'].isna() | (snigur['Дата начала (Снигур)'] <= as_of))
        & (sn_end.isna() | (sn_end >= as_of))
    )

status = snigur['Статус (Снигур)'].fillna('').str.lower()
closed_dt = snigur['Дата закрытия (Снигур)']
conflict = pd.Series(None, index=snigur.index, dtype=object)
conflict[status.str.contains('закрыт') & ~status.str.contains('на закрыт') & closed_dt.isna()] = 'Статус «Закрыт», даты закрытия нет'
conflict[status.str.contains('работает') & closed_dt.notna() & (closed_dt < as_of)] = 'Статус «Работает», но закрыт до %s' % as_of_ru
conflict[snigur['Дата начала (Снигур)'].isna()] = 'Нет даты начала'
snigur['Противоречие статуса'] = conflict

sn_rows = snigur.groupby('agr_key').size().rename('Строк у Снигура')
snigur_agr = (
    snigur.sort_values(['agr_key', 'Действует на дату', 'Конец (Снигур)'], ascending=[True, False, False], na_position='first')
    .drop_duplicates('agr_key')
    .merge(sn_rows, on='agr_key')
    .reset_index(drop=True)
)
snigur_active = snigur_agr[snigur_agr['Действует на дату']].reset_index(drop=True)

print('Договоров у Снигура:', len(snigur_agr))
print('Действуют на %s:' % as_of_ru, len(snigur_active))
print('Номеров с повтором:', int((snigur_agr['Строк у Снигура'] > 1).sum()))
display(pd.crosstab(snigur_agr['Статус (Снигур)'].fillna('(пусто)'), snigur_agr['Действует на дату'], margins=True))
display(snigur_agr['Противоречие статуса'].value_counts(dropna=False))

agr_status = snigur_agr['Статус (Снигур)'].fillna('').str.lower()
is_closed = agr_status.str.contains('закрыт') & ~agr_status.str.contains('на закрыт')
closed_by_month = (
    snigur_agr.loc[is_closed, 'Дата закрытия (Снигур)'].dt.strftime('%Y-%m').fillna('(пусто)')
    .value_counts().sort_index()
    .rename_axis('«Закрыт» у Снигура: месяц закрытия').reset_index(name='Договоров')
)
close_by_status = (
    snigur_agr.assign(**{
        'Статус у Снигура: топ-5 дат закрытия': snigur_agr['Статус (Снигур)'].fillna('(пусто)'),
        'Дата закрытия': snigur_agr['Дата закрытия (Снигур)'].dt.strftime('%d.%m.%Y').fillna('(пусто)'),
    })
    .groupby(['Статус у Снигура: топ-5 дат закрытия', 'Дата закрытия'])
    .size().rename('Договоров').reset_index()
    .sort_values(['Статус у Снигура: топ-5 дат закрытия', 'Договоров'], ascending=[True, False])
    .groupby('Статус у Снигура: топ-5 дат закрытия').head(5)
    .reset_index(drop=True)
)
display(closed_by_month)
display(close_by_status)


## 3. Сравнение по номеру договора

In [ ]:
cmp = snigur_active.merge(mine, on='agr_key', how='outer', indicator=True)

both = cmp[cmp['_merge'] == 'both'].copy()

def date_diff(row):
    out = []
    if not (pd.isna(row['Дата начала (Снигур)']) and pd.isna(row['Дата открытия (мой)'])) \
            and row['Дата начала (Снигур)'] != row['Дата открытия (мой)']:
        out.append('дата начала')
    if not (pd.isna(row['Дата окончания (Снигур)']) and pd.isna(row['Дата закрытия (мой)'])) \
            and row['Дата окончания (Снигур)'] != row['Дата закрытия (мой)']:
        out.append('дата окончания')
    return ', '.join(out) or None

both['Даты расходятся'] = both.apply(date_diff, axis=1) if len(both) else None

only_snigur = cmp[cmp['_merge'] == 'left_only'].copy()

only_mine = cmp[cmp['_merge'] == 'right_only'][['agr_key'] + [c for c in mine.columns if c != 'agr_key']].copy()
sn_inactive = snigur_agr.loc[~snigur_agr['Действует на дату']].drop(columns=['Действует на дату'])
only_mine = only_mine.merge(sn_inactive, on='agr_key', how='left')

print('Совпало:', len(both), '| из них даты расходятся:', int(both['Даты расходятся'].notna().sum()))
print('Только у Снигура:', len(only_snigur))
print('Только у меня:', len(only_mine), '| из них у Снигура есть, но не действует:', int(only_mine['Статус (Снигур)'].notna().sum()))


### 3a. Какие даты расходятся у совпавших договоров

Дата начала: разница в днях (моя минус Снигура). Дата окончания: моя `Дата закрытия договора` против «Даты окончания действия договора» у Снигура.


In [ ]:
both['Разница начала, дней'] = (both['Дата открытия (мой)'] - both['Дата начала (Снигур)']).dt.days
both['Разница окончания, дней'] = (both['Дата закрытия (мой)'] - both['Дата окончания (Снигур)']).dt.days

def end_pattern(r):
    sn, my = r['Дата окончания (Снигур)'], r['Дата закрытия (мой)']
    if pd.isna(sn) and pd.isna(my):
        return 'обе пустые'
    if pd.isna(sn):
        return 'у Снигура пусто, у меня %s' % ('условная дата (2100+)' if my.year >= 2100 else 'есть дата')
    if pd.isna(my):
        return 'у меня пусто, у Снигура есть дата'
    return 'совпадают' if sn == my else 'обе есть, разные'

date_kinds = (
    both['Даты расходятся'].fillna('совпадают')
    .value_counts().rename_axis('Совпавшие договоры: что расходится').reset_index(name='Договоров')
)
end_patterns = (
    both.apply(end_pattern, axis=1).value_counts()
    .rename_axis('Дата окончания: Снигур против моей').reset_index(name='Договоров')
) if len(both) else pd.DataFrame(columns=['Дата окончания: Снигур против моей', 'Договоров'])
start_diff_top = (
    both['Разница начала, дней'].map(lambda v: 'нет даты' if pd.isna(v) else int(v))
    .value_counts().head(10)
    .rename_axis('Дата начала: разница, дней (топ-10)').reset_index(name='Договоров')
)
display(date_kinds)
display(start_diff_top)
display(end_patterns)

show_cols = ['Номер договора (Снигур)', 'Дата начала (Снигур)', 'Дата открытия (мой)', 'Разница начала, дней',
             'Дата окончания (Снигур)', 'Дата закрытия (Снигур)', 'Дата закрытия (мой)', 'Статус (Снигур)']
for kind in ('дата начала', 'дата окончания'):
    part = both[both['Даты расходятся'].fillna('').str.contains(kind)]
    print('Примеры, где расходится %s (%s):' % (kind, len(part)))
    display(part[show_cols].head(5))

def digits_key(k):
    d = re.sub(r'\D', '', k or '').lstrip('0')
    return d if len(d) >= 4 else None

def unique_digits(df, num_col, extra):
    out = df[['agr_key', num_col] + extra].assign(digits=df['agr_key'].map(digits_key)).dropna(subset=['digits'])
    return out[~out['digits'].duplicated(keep=False)]

num_pairs = (
    unique_digits(only_snigur, 'Номер договора (Снигур)', ['Клиент (Снигур)', 'Статус (Снигур)'])
    .merge(unique_digits(only_mine, 'Номер договора (мой)', ['ИНН', 'ЦФТ ID']), on='digits', suffixes=('_sn', '_my'))
    .rename(columns={'digits': 'Цифры номера'})
)
sn_pair = num_pairs.set_index('agr_key_sn')['Номер договора (мой)']
my_pair = num_pairs.set_index('agr_key_my')['Номер договора (Снигур)']
only_snigur['Пара по цифрам номера'] = only_snigur['agr_key'].map(sn_pair)
only_mine['Пара по цифрам номера'] = only_mine['agr_key'].map(my_pair)
num_pairs = num_pairs[['Номер договора (Снигур)', 'Номер договора (мой)', 'Цифры номера', 'Клиент (Снигур)',
                       'Статус (Снигур)', 'ИНН', 'ЦФТ ID']].sort_values('Номер договора (Снигур)')
print('Возможно один договор с разной записью номера:', len(num_pairs))
display(num_pairs.head(20))


## 4. Причины расхождений по `ods_alpha`

Для каждого номера берём все `n_agr` с таким `c_agr_number` и смотрим, на каком фильтре моей выгрузки он отваливается. Если номеру соответствует несколько `n_agr`, берётся тот, что прошёл дальше всех.

`scd1_agreements` с клиентами и привязки `P` читаются из озера один раз целиком (два запроса), дальше всё считается в pandas. Фильтр по списку номеров через `regexp_replace` заставлял сканировать всю таблицу на каждую порцию номеров.


In [ ]:
import time

def timed_fetch(label, sql):
    t0 = time.time()
    print('%s: запрос пошёл в %s' % (label, time.strftime('%H:%M:%S')), flush=True)
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        df = imp.fetch(sql)
    df.columns = [str(c).split('.')[-1].lower() for c in df.columns]
    print('%s: %s строк за %.0f с' % (label, len(df), time.time() - t0), flush=True)
    return df

def id_str(s):
    return s.astype('string').str.strip().str.replace(r'\.0+$', '', regex=True)

lake_agr = timed_fetch('scd1_agreements + scd1_companies', '''
SELECT
  upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) AS agr_key,
  cast(a.n_agr AS string) AS n_agr,
  a.c_agr_number,
  a.acq_class,
  a.d_valid_from,
  a.d_valid_to,
  coalesce(a.ods_deleted_flg, '0') AS agr_deleted,
  regexp_replace(c.c_inn, '[^0-9]', '') AS inn,
  c.c_cmp_name AS cmp_name,
  coalesce(c.ods_deleted_flg, '0') AS cmp_deleted
FROM ods_alpha.scd1_agreements a
LEFT JOIN ods_alpha.scd1_companies c ON c.n_cmp = a.n_cmp_client
WHERE a.c_agr_number IS NOT NULL
''')
lake_p = timed_fetch('scd1_agr_terms, привязки P', f'''
SELECT
  cast(t.n_agr AS string) AS n_agr,
  1 AS has_p_any,
  max(CASE WHEN t.d_valid_from <= '{as_of_sql}'
            AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{as_of_sql}')
           THEN 1 ELSE 0 END) AS has_p_asof
FROM ods_alpha.scd1_agr_terms t
WHERE upper(trim(t.cf_ter_type)) = 'P'
  AND coalesce(t.ods_deleted_flg, '0') <> '1'
GROUP BY t.n_agr
''')
lake_agr['n_agr'] = id_str(lake_agr['n_agr'])
lake_agr['agr_deleted'] = lake_agr['agr_deleted'].astype(str)
lake_agr['cmp_deleted'] = lake_agr['cmp_deleted'].astype(str)
lake_p['n_agr'] = id_str(lake_p['n_agr'])
lake_p = lake_p.drop_duplicates('n_agr')

diff_keys = sorted(set(only_snigur['agr_key']) | set(only_mine['agr_key']))
ods = (
    lake_agr[lake_agr['agr_key'].isin(diff_keys)]
    .drop(columns=['c_agr_number', 'cmp_name'])
    .merge(lake_p, on='n_agr', how='left')
)
ods[['has_p_any', 'has_p_asof']] = ods[['has_p_any', 'has_p_asof']].fillna(0).astype(int)
ods['d_valid_from'] = pd.to_datetime(ods['d_valid_from'], errors='coerce').dt.normalize()
ods['d_valid_to'] = pd.to_datetime(ods['d_valid_to'], errors='coerce').dt.normalize()
ods = ods.reset_index(drop=True)
print('Номеров в расхождениях:', len(diff_keys), '| найдено строк в ods_alpha:', len(ods))


In [ ]:
def fmt(d):
    return '' if pd.isna(d) else d.strftime('%d.%m.%Y')

def ods_stage(r):
    if str(r['agr_deleted']) == '1':
        return 1, 'Договор удалён в ods_alpha'
    if str(r['acq_class'] or '').strip().upper() != 'SA':
        return 2, 'acq_class = %s, не SA' % (r['acq_class'] if pd.notna(r['acq_class']) else 'пусто')
    if pd.notna(r['d_valid_from']) and r['d_valid_from'] > as_of:
        return 3, 'По ods_alpha договор начался позже %s' % as_of_ru
    if pd.notna(r['d_valid_to']) and r['d_valid_to'] < as_of:
        return 3, 'По ods_alpha договор закрыт до %s' % as_of_ru
    if int(r['has_p_asof']) != 1:
        if int(r['has_p_any']) == 1:
            return 4, 'Привязка P есть, но не действует на %s' % as_of_ru
        return 4, 'Привязки P нет'
    if str(r['cmp_deleted']) == '1' or pd.isna(r['inn']) or not r['inn']:
        return 5, 'Нет клиента или ИНН в ods_alpha.scd1_companies'
    return 6, 'Проходит все фильтры моей выгрузки'

if len(ods):
    staged = ods.apply(ods_stage, axis=1, result_type='expand')
    ods['stage'], ods['ods_reason'] = staged[0], staged[1]
else:
    ods['stage'], ods['ods_reason'] = pd.Series(dtype=int), pd.Series(dtype=object)
n_agr_cnt = ods.groupby('agr_key')['n_agr'].nunique().rename('n_agr в ods_alpha')
ods_best = (
    ods.sort_values(['agr_key', 'stage'], ascending=[True, False])
    .drop_duplicates('agr_key')
    .merge(n_agr_cnt, on='agr_key')
    .rename(columns={
        'n_agr': 'n_agr (ods_alpha)',
        'acq_class': 'acq_class (ods_alpha)',
        'd_valid_from': 'Дата начала (ods_alpha)',
        'd_valid_to': 'Дата окончания (ods_alpha)',
        'inn': 'ИНН (ods_alpha)',
        'has_p_asof': 'Привязка P на дату',
    })
    [['agr_key', 'n_agr (ods_alpha)', 'n_agr в ods_alpha', 'acq_class (ods_alpha)', 'Дата начала (ods_alpha)',
      'Дата окончания (ods_alpha)', 'ИНН (ods_alpha)', 'Привязка P на дату', 'ods_reason']]
)

only_snigur = only_snigur.merge(ods_best, on='agr_key', how='left')
only_snigur['Причина'] = only_snigur['ods_reason'].fillna('Договора нет в ods_alpha')
only_snigur.loc[only_snigur['Причина'] == 'Проходит все фильтры моей выгрузки', 'Причина'] = (
    'Проходит все фильтры, но нет в моей выгрузке - проверить вручную'
)

def mine_reason(r):
    if pd.notna(r['Статус (Снигур)']) or pd.notna(r['Конец (Снигур)']):
        return 'У Снигура не действует на %s: статус %s, закрыт/окончен %s' % (
            as_of_ru, r['Статус (Снигур)'] if pd.notna(r['Статус (Снигур)']) else 'пусто', fmt(r['Конец (Снигур)']))
    return 'Договора нет в выгрузке Снигура'

only_mine = only_mine.merge(ods_best, on='agr_key', how='left')
only_mine['Причина'] = only_mine.apply(mine_reason, axis=1) if len(only_mine) else None

only_snigur.loc[only_snigur['Пара по цифрам номера'].notna(), 'Причина'] = 'Возможно, есть у меня под другим номером'
only_mine.loc[only_mine['Пара по цифрам номера'].notna(), 'Причина'] = 'Возможно, есть у Снигура под другим номером'

closed_years = (
    only_snigur.loc[only_snigur['Причина'] == 'По ods_alpha договор закрыт до %s' % as_of_ru, 'Дата окончания (ods_alpha)']
    .dt.year.value_counts().sort_index()
    .rename_axis('Закрыт в ods_alpha: год закрытия').reset_index(name='Договоров')
)

reason_by_status = (
    pd.crosstab(only_snigur['Причина'], only_snigur['Статус (Снигур)'].fillna('(пусто)'), margins=True, margins_name='Всего')
    .rename_axis(index='Только у Снигура: причина × статус', columns=None).reset_index()
)

closed_reason = 'По ods_alpha договор закрыт до %s' % as_of_ru
closed_sn = only_snigur[only_snigur['Причина'] == closed_reason]

def end_vs_ods_kind(r):
    sn, ods_to = r['Дата окончания (Снигур)'], r['Дата окончания (ods_alpha)']
    if pd.isna(sn):
        return 'у Снигура «Дата окончания» пустая'
    if sn == ods_to:
        return 'совпадает с датой закрытия в ods_alpha'
    if sn < as_of:
        return 'у Снигура тоже раньше %s, но другая' % as_of_ru
    return 'у Снигура не раньше %s' % as_of_ru

end_vs_ods = (
    (closed_sn.apply(end_vs_ods_kind, axis=1) if len(closed_sn) else pd.Series(dtype=object))
    .value_counts().rename_axis('Закрыт в ods_alpha: «Дата окончания» у Снигура').reset_index(name='Договоров')
)
closed_months_2026 = (
    closed_sn.loc[closed_sn['Дата окончания (ods_alpha)'].dt.year == as_of.year, 'Дата окончания (ods_alpha)']
    .dt.strftime('%Y-%m').value_counts().sort_index()
    .rename_axis('Закрыт в ods_alpha в %s: месяц' % as_of.year).reset_index(name='Договоров')
)
sn_end_passed = snigur_agr['Дата окончания (Снигур)'].notna() & (snigur_agr['Дата окончания (Снигур)'] < as_of)
print('У Снигура «Дата окончания» раньше %s, но по дате закрытия действует: %s' % (as_of_ru, int(sn_end_passed.sum())))

display(only_snigur['Причина'].value_counts())
display(closed_years)
display(reason_by_status)
display(end_vs_ods)
display(closed_months_2026)
display(only_mine['Причина'].str.replace(r':.*', '', regex=True).value_counts())


## 4b. Тот же клиент под другим номером

Для всех договоров Снигура и моих поднимаем из `ods_alpha` ИНН и наименование клиента. У моих договоров ИНН берётся из моей выгрузки, у Снигура - из `ods_alpha` (если договор там есть). Затем для каждого расхождения ищем того же клиента на другой стороне: сначала по ИНН, затем по наименованию без ОПФ, кавычек и знаков препинания.

- «его другой договор есть у обоих» - клиент у другой стороны есть, но с договором, который есть и у меня, и у Снигура. Похоже, что другая сторона показывает не все договоры клиента.
- «под другим номером, которого нет ...» - похоже на перезаключение договора или разную запись номера. Колонка «новее» показывает, начался ли договор другой стороны позже.


In [ ]:
LEGAL_FORMS = [
    'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ', 'НЕПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО',
    'АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ИНДИВИДУАЛЬНЫЙ ПРЕДПРИНИМАТЕЛЬ', 'ГЛАВА КФХ',
    'ООО', 'ОАО', 'ЗАО', 'ПАО', 'НАО', 'АО', 'ИП', 'МБУ', 'МАУ', 'МБОУ', 'МАОУ', 'МБДОУ', 'МАДОУ', 'МКУ', 'ГБУ',
    'ГАУ', 'ГКУ', 'ГБУЗ', 'ФГБУ', 'ФГУП', 'МУП', 'ГУП', 'АНО', 'ТСЖ', 'СНТ', 'КФХ',
]
LEGAL_RE = re.compile(r'\b(%s)\b' % '|'.join(LEGAL_FORMS))

def norm_name(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).upper().replace('Ё', 'Е')
    s = re.sub(r'[^0-9A-ZА-Я]+', ' ', s)
    s = LEGAL_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None

def norm_inn(v):
    if v is None or pd.isna(v):
        return None
    s = re.sub(r'\D', '', str(v).split('.')[0])
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s or None

all_keys = set(snigur_agr['agr_key']) | set(mine['agr_key'])
cmp_info = (
    lake_agr.loc[lake_agr['agr_key'].isin(all_keys) & lake_agr['cmp_deleted'].ne('1'), ['agr_key', 'inn', 'cmp_name']]
    .drop_duplicates().copy()
)
cmp_info['inn'] = cmp_info['inn'].map(norm_inn)
names_by_key = cmp_info.dropna(subset=['cmp_name']).drop_duplicates('agr_key').set_index('agr_key')['cmp_name']
inn_by_key = cmp_info.dropna(subset=['inn']).drop_duplicates('agr_key').set_index('agr_key')['inn']

both_keys = set(both['agr_key'])
sn_side = snigur_agr[['agr_key', 'Номер договора (Снигур)', 'Клиент (Снигур)', 'Статус (Снигур)', 'Дата начала (Снигур)']].merge(
    cmp_info[['agr_key', 'inn']].dropna().drop_duplicates(), on='agr_key', how='left')
sn_side['name_norm'] = sn_side['Клиент (Снигур)'].map(norm_name)
my_side = mine[['agr_key', 'Номер договора (мой)', 'ИНН', 'Дата открытия (мой)']].copy()
my_side['inn'] = my_side['ИНН'].map(norm_inn)
my_side['name_norm'] = my_side['agr_key'].map(names_by_key).map(norm_name)

def index_by(df, col):
    return {k: g for k, g in df.dropna(subset=[col]).groupby(col)}

sn_by_inn, sn_by_name = index_by(sn_side, 'inn'), index_by(sn_side, 'name_norm')
my_by_inn, my_by_name = index_by(my_side, 'inn'), index_by(my_side, 'name_norm')

def find_same_client(key, inn, name, by_inn, by_name):
    for how, idx, val in (('по ИНН', by_inn, inn), ('по наименованию', by_name, name)):
        if val and val in idx:
            cand = idx[val]
            cand = cand[cand['agr_key'] != key].drop_duplicates('agr_key')
            if len(cand):
                return how, cand
    return None, None

def same_client(key, inn, name, start, by_inn, by_name, num_col, start_col, status_col, other, me):
    how, cand = find_same_client(key, inn, name, by_inn, by_name)
    if how is None:
        return 'Клиента у %s нет ни по ИНН, ни по наименованию' % other, None, None
    cand = cand.sort_values(start_col, ascending=False)
    in_both = cand['agr_key'].isin(both_keys)
    if in_both.all():
        kind = 'Клиент есть у %s (%s), его другой договор есть у обоих' % (other, how)
    else:
        kind = 'Клиент есть у %s (%s) под другим номером, которого нет у %s' % (other, how, me)
    items = []
    for c in cand.head(5).to_dict('records'):
        extra = [fmt(c[start_col]) and 'с ' + fmt(c[start_col])]
        if status_col:
            extra.insert(0, c[status_col] if pd.notna(c[status_col]) else 'статус пуст')
        if c['agr_key'] in both_keys:
            extra.append('есть у обоих')
        items.append('%s (%s)' % (c[num_col], ', '.join(e for e in extra if e)))
    newer = None
    if pd.notna(start) and cand[start_col].notna().any():
        newer = 'да' if (cand[start_col] > start).any() else 'нет'
    return kind, '; '.join(items), newer

res = [
    same_client(r['agr_key'], norm_inn(r['ИНН']), norm_name(names_by_key.get(r['agr_key'])), r['Дата открытия (мой)'],
                sn_by_inn, sn_by_name, 'Номер договора (Снигур)', 'Дата начала (Снигур)', 'Статус (Снигур)', 'Снигура', 'меня')
    for r in only_mine.to_dict('records')
]
only_mine['Тот же клиент у Снигура'] = [x[0] for x in res]
only_mine['Договоры клиента у Снигура'] = [x[1] for x in res]
only_mine['У Снигура договор новее моего'] = [x[2] for x in res]
only_mine['Наименование (ods_alpha)'] = only_mine['agr_key'].map(names_by_key)
no_sn = only_mine['Причина'] == 'Договора нет в выгрузке Снигура'
only_mine.loc[no_sn, 'Причина'] = only_mine.loc[no_sn, 'Тот же клиент у Снигура']

res = [
    same_client(r['agr_key'], inn_by_key.get(r['agr_key']), norm_name(r['Клиент (Снигур)']), r['Дата начала (Снигур)'],
                my_by_inn, my_by_name, 'Номер договора (мой)', 'Дата открытия (мой)', None, 'меня', 'Снигура')
    for r in only_snigur.to_dict('records')
]
only_snigur['Тот же клиент у меня'] = [x[0] for x in res]
only_snigur['Договоры клиента у меня'] = [x[1] for x in res]
only_snigur['Мой договор новее'] = [x[2] for x in res]

sn_contracts = sn_side.drop_duplicates('agr_key')
per_client = pd.DataFrame([
    ('Договоров у Снигура (все статусы)', len(sn_contracts)),
    ('  из них с ИНН из ods_alpha', int(sn_contracts['inn'].notna().sum())),
    ('Клиентов у Снигура по ИНН', sn_side['inn'].nunique()),
    ('Клиентов у Снигура по наименованию', sn_contracts['name_norm'].nunique()),
    ('Клиентов у Снигура с 2+ договорами (по наименованию)', int((sn_contracts.groupby('name_norm').size() >= 2).sum())),
    ('Договоров у меня', len(my_side)),
    ('Клиентов у меня по ИНН', my_side['inn'].nunique()),
    ('Клиентов у меня с 2+ договорами (по ИНН)', int((my_side.groupby('inn').size() >= 2).sum())),
], columns=['Договоров на клиента', 'Значение'])

same_client_mine = (
    only_mine['Тот же клиент у Снигура'].value_counts()
    .rename_axis('Только у меня: тот же клиент у Снигура').reset_index(name='Договоров')
)
same_client_sn = (
    pd.crosstab(only_snigur['Тот же клиент у меня'], only_snigur['Причина'], margins=True, margins_name='Всего')
    .rename_axis(index='Только у Снигура: тот же клиент у меня × причина', columns=None).reset_index()
)
newer_mine = (
    only_mine.loc[only_mine['Тот же клиент у Снигура'].str.contains('под другим номером'), 'У Снигура договор новее моего']
    .fillna('нет дат').value_counts()
    .rename_axis('Только у меня, клиент у Снигура под другим номером: у Снигура новее').reset_index(name='Договоров')
)
display(per_client)
display(same_client_mine)
display(newer_mine)
display(same_client_sn)
display(only_mine.loc[only_mine['Тот же клиент у Снигура'].str.contains('под другим номером'),
                      ['Номер договора (мой)', 'ИНН', 'Наименование (ods_alpha)', 'Дата открытия (мой)',
                       'Договоры клиента у Снигура', 'У Снигура договор новее моего']].head(10))


### 4c. Чем «клиента у Снигура нет» отличаются от совпавших

Сравниваем договоры «только у меня, клиента у Снигура нет ни по ИНН, ни по наименованию» с договорами, которые есть у обоих: пустой ЦФТ ID, РФ обслуживания, год и месяц открытия. Если доли сильно расходятся, расхождение сидит в этой группе (например, договоры без ЦФТ ID или один РФ).


In [ ]:
no_sn_client = only_mine['Тот же клиент у Снигура'].str.startswith('Клиента у Снигура нет')
grp_frames = {'Клиента у Снигура нет': only_mine[no_sn_client], 'Есть у обоих': both}
print({g: len(d) for g, d in grp_frames.items()})

def blank(s):
    s = s.astype('string').str.strip()
    return s.isna() | s.str.lower().isin(['', 'nan', 'none', '<na>'])

def share_table(title, values_of, top=None, by_value=False):
    cols = {g + ': договоров': values_of(d).astype('string').fillna('(пусто)').value_counts() for g, d in grp_frames.items()}
    t = pd.DataFrame(cols).fillna(0).astype(int)
    t = t.sort_index() if by_value else t.sort_values(list(cols)[0], ascending=False)
    if top and len(t) > top:
        t = pd.concat([t.iloc[:top], t.iloc[top:].sum().to_frame('Прочие').T])
    for g in grp_frames:
        n = t[g + ': договоров'].sum()
        t[g + ': %'] = (t[g + ': договоров'] / n * 100).round(1) if n else 0.0
    t = t[[c for g in grp_frames for c in (g + ': договоров', g + ': %')]]
    return t.rename_axis(title).reset_index()

open_dt = lambda d: d['Дата открытия (мой)']
nosn_cft = share_table('Нет у Снигура / есть у обоих: ЦФТ ID',
                       lambda d: blank(d['ЦФТ ID']).map({True: 'ЦФТ ID пустой', False: 'ЦФТ ID есть'}))
nosn_rf = share_table('Нет у Снигура / есть у обоих: РФ', lambda d: d['РФ'], top=15)
nosn_year = share_table('Нет у Снигура / есть у обоих: год открытия',
                        lambda d: open_dt(d).dt.year.astype('Int64'), by_value=True)
nosn_month = share_table('Нет у Снигура / есть у обоих: месяц открытия с 2025',
                         lambda d: open_dt(d).dt.strftime('%Y-%m').where((open_dt(d).dt.year >= 2025) | open_dt(d).isna(), 'до 2025'),
                         by_value=True)
for t in (nosn_cft, nosn_rf, nosn_year, nosn_month):
    display(t)


### 4d. Нечёткий поиск по наименованию

В 4b наименование сравнивалось точно (после чистки ОПФ и знаков). Здесь для договоров «клиента у Снигура нет» ищем похожее наименование среди всех договоров Снигура:
1. **ИП по фамилии и инициалам** - «ИВАНОВ ИВАН ИВАНОВИЧ» и «ИВАНОВ И.И.»;
2. **по словам** - все значимые слова (от 3 букв) одного имени есть в другом: лишние слова, другой порядок;
3. **похожесть строк** - `difflib` от `FUZZY_MIN`, сравнение с именами Снигура на ту же первую букву самого длинного слова.

Латинские буквы, похожие на кириллицу, перед сравнением заменяются. Если нашлись сотни - это те же клиенты с другой записью имени. Если единицы - клиентов действительно нет в выгрузке Снигура.

Для ручной проверки - 10 случайных договоров: искать в файле Снигура Ctrl+F по «Слову для поиска».


In [ ]:
import difflib

FUZZY_MIN = 0.85
N_MANUAL = 10
LAT2CYR = str.maketrans('ABCEHKMOPTXY', 'АВСЕНКМОРТХУ')

def fz_norm(v):
    if v is None or pd.isna(v):
        return None
    return norm_name(str(v).upper().translate(LAT2CYR))

def is_ip(v):
    return v is not None and not pd.isna(v) and bool(re.search(r'\bИП\b|ИНДИВИДУАЛЬН', str(v).upper()))

def ip_key(name):
    toks = name.split() if name else []
    if 2 <= len(toks) <= 3 and len(toks[0]) >= 3:
        return toks[0] + ' ' + ' '.join(t[0] for t in toks[1:])
    return None

def sig(name):
    return {t for t in name.split() if len(t) >= 3} if name else set()

def longest(name):
    return max(name.split(), key=len) if name else None

sn_names = snigur_agr[['Клиент (Снигур)', 'Номер договора (Снигур)', 'Статус (Снигур)']].copy()
sn_names['fz'] = sn_names['Клиент (Снигур)'].map(fz_norm)
sn_names = sn_names.dropna(subset=['fz']).reset_index(drop=True)
sn_names['ip_key'] = sn_names['fz'].map(ip_key)
sn_names['sig'] = sn_names['fz'].map(sig)
sn_names['first'] = sn_names['fz'].map(lambda s: longest(s)[0])

by_ip = sn_names.dropna(subset=['ip_key']).groupby('ip_key').groups
by_token = {}
for i, toks in sn_names['sig'].items():
    for t in toks:
        by_token.setdefault(t, set()).add(i)
by_first = sn_names.groupby('first').groups

def ratio(a, b):
    m = difflib.SequenceMatcher(None, a, b)
    return m.ratio() if m.real_quick_ratio() >= FUZZY_MIN and m.quick_ratio() >= FUZZY_MIN else 0.0

def best_of(name, idx):
    best_i, best_r = None, -1.0
    for i in idx:
        r = difflib.SequenceMatcher(None, name, sn_names.at[i, 'fz']).ratio()
        if r > best_r:
            best_i, best_r = i, r
    return (best_i, best_r) if best_i is not None else (None, None)

def fuzzy_find(raw):
    name = fz_norm(raw)
    if not name:
        return 'нет наименования', None, None
    if is_ip(raw) and ip_key(name) in by_ip:
        i, r = best_of(name, by_ip[ip_key(name)])
        return 'ИП по фамилии и инициалам', i, r
    my_sig = sig(name)
    cand = set()
    if my_sig:
        sets = [by_token.get(t, set()) for t in my_sig]
        cand = set.intersection(*sets) if all(sets) else set()
        for t in my_sig:
            cand |= {i for i in by_token.get(t, ()) if sn_names.at[i, 'sig'] and sn_names.at[i, 'sig'] <= my_sig}
    small = lambda i: min(len(my_sig), len(sn_names.at[i, 'sig']))
    cand = {i for i in cand if small(i) >= 2 or max(map(len, my_sig & sn_names.at[i, 'sig']), default=0) >= 6}
    if cand:
        i, r = best_of(name, cand)
        return 'по словам', i, r
    block = by_first.get(longest(name)[0], [])
    hits = [(ratio(name, sn_names.at[i, 'fz']), i) for i in block]
    hits = [h for h in hits if h[0] >= FUZZY_MIN]
    if hits:
        r, i = max(hits)
        return 'похожесть строк', i, r
    i, r = best_of(name, block) if len(block) else (None, None)
    return 'не нашлось', i, r

fz_group = only_mine['Тот же клиент у Снигура'].str.startswith('Клиента у Снигура нет')
res = {ix: fuzzy_find(only_mine.at[ix, 'Наименование (ods_alpha)']) for ix in only_mine.index[fz_group]}
only_mine['Как найден (нечётко)'] = pd.Series({ix: r[0] for ix, r in res.items()}, dtype=object)
cand_i = pd.Series({ix: r[1] for ix, r in res.items()}, dtype=object)
for col, src in (('Похожий клиент у Снигура', 'Клиент (Снигур)'), ('Номер у Снигура', 'Номер договора (Снигур)'),
                 ('Статус у Снигура', 'Статус (Снигур)')):
    only_mine[col] = cand_i.map(lambda i: sn_names.at[i, src] if i is not None and not pd.isna(i) else None)
only_mine['Похожесть'] = pd.Series({ix: round(r[2], 3) if r[2] is not None else None for ix, r in res.items()}, dtype=float)
found_fz = only_mine['Как найден (нечётко)'].isin(['ИП по фамилии и инициалам', 'по словам', 'похожесть строк'])
only_mine.loc[~found_fz, ['Похожий клиент у Снигура', 'Номер у Снигура', 'Статус у Снигура']] = None

fuzzy_sum = (
    only_mine.loc[fz_group, 'Как найден (нечётко)'].value_counts()
    .rename_axis('Клиента у Снигура нет: нечёткий поиск').reset_index(name='Договоров')
)
fuzzy_status = (
    only_mine.loc[fz_group & found_fz, 'Статус у Снигура'].fillna('(пусто)').value_counts()
    .rename_axis('Нашлись нечётко: статус у Снигура').reset_index(name='Договоров')
)
fuzzy_cols = ['Номер договора (мой)', 'ИНН', 'Наименование (ods_alpha)', 'Дата открытия (мой)', 'Как найден (нечётко)',
              'Похожесть', 'Похожий клиент у Снигура', 'Номер у Снигура', 'Статус у Снигура']
fuzzy_out = only_mine.loc[fz_group, fuzzy_cols].sort_values(['Как найден (нечётко)', 'Похожесть'], ascending=[True, False])

manual = only_mine.loc[fz_group].sample(min(N_MANUAL, int(fz_group.sum())), random_state=7)
manual_out = manual[['Номер договора (мой)', 'ИНН', 'Наименование (ods_alpha)']].copy()
manual_out['Слово для поиска'] = [
    (fz_norm(n) or '').split()[0] if is_ip(n) and fz_norm(n) else longest(fz_norm(n))
    for n in manual['Наименование (ods_alpha)']]
best = [best_of(fz_norm(n), by_first.get(longest(fz_norm(n))[0], [])) if fz_norm(n) else (None, None)
        for n in manual['Наименование (ods_alpha)']]
manual_out['Ближайшее у Снигура'] = [sn_names.at[i, 'Клиент (Снигур)'] if i is not None else None for i, _ in best]
manual_out['Похожесть ближайшего'] = [round(r, 3) if r is not None else None for _, r in best]
manual_out['Как найден (нечётко)'] = manual['Как найден (нечётко)'].values
manual_out['Нашёл вручную (заполнить)'] = ''

display(fuzzy_sum)
display(fuzzy_status)
display(fuzzy_out[found_fz.reindex(fuzzy_out.index)].head(15))
display(manual_out)


## 5. Разбор примеров: есть у Снигура, нет у меня

Берём `N_EXAMPLES` договоров из «Только у Снигура», по кругу по причинам, чтобы попали разные случаи. Договоры, у которых нашлась пара у меня по цифрам номера, в примеры не берутся. Конкретные номера можно задать в `EXAMPLE_AGRS`.

По ним из `ods_alpha` поднимаем всё без фильтров: строки договора (в том числе удалённые), клиента, всю историю `scd1_agr_terms`. Затем проверяем по шагам моей сборки (блок `sa` в `trade_acquiring_clients_month_excel.ipynb`):

1. есть строка в `scd1_agreements` с таким номером;
2. строка договора не удалена;
3. `acq_class = 'SA'`;
4. договор действует на дату;
5. есть клиент в `scd1_companies`, он не удалён, и у него заполнен ИНН;
6. есть неудалённая привязка `P`, действующая на дату.

Если номер не найден точно, ищем по цифрам из номера: так видно, если договор записан в другом формате.


In [ ]:
N_EXAMPLES = 10
EXAMPLE_AGRS = []

if EXAMPLE_AGRS:
    ex_keys = [k for k in (norm_agr(a) for a in EXAMPLE_AGRS) if k]
else:
    groups = [
        g['agr_key'].tolist()
        for _, g in sorted(
            only_snigur[only_snigur['Пара по цифрам номера'].isna()]
            .sort_values('Номер договора (Снигур)').groupby('Причина'),
            key=lambda kv: -len(kv[1]),
        )
    ]
    ex_keys, i = [], 0
    while len(ex_keys) < N_EXAMPLES and any(i < len(g) for g in groups):
        for g in groups:
            if i < len(g) and len(ex_keys) < N_EXAMPLES:
                ex_keys.append(g[i])
        i += 1
print('Примеров:', len(ex_keys))
print(ex_keys)

def sql_in(values):
    return ', '.join("'%s'" % str(v).replace("'", "''") for v in values)

def agr_detail_sql(where):
    return f'''
SELECT
  upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) AS ods_key,
  regexp_replace(a.c_agr_number, '[^0-9]', '') AS ods_digits,
  cast(a.n_agr AS string) AS n_agr,
  a.c_agr_number,
  a.acq_class,
  a.d_valid_from,
  a.d_valid_to,
  coalesce(a.ods_deleted_flg, '0') AS agr_deleted,
  cast(a.n_cmp_client AS string) AS n_cmp_client,
  CASE WHEN c.n_cmp IS NULL THEN 0 ELSE 1 END AS cmp_found,
  c.c_inn,
  coalesce(c.ods_deleted_flg, '0') AS cmp_deleted
FROM ods_alpha.scd1_agreements a
LEFT JOIN ods_alpha.scd1_companies c ON c.n_cmp = a.n_cmp_client
WHERE {where}
'''

def terms_sql(n_agrs):
    return f'''
SELECT
  cast(t.n_agr AS string) AS n_agr,
  t.cf_ter_type,
  t.d_valid_from,
  t.d_valid_to,
  coalesce(t.ods_deleted_flg, '0') AS term_deleted
FROM ods_alpha.scd1_agr_terms t
WHERE cast(t.n_agr AS string) IN ({sql_in(n_agrs)})
'''

agr_cols = ['ods_key', 'ods_digits', 'n_agr', 'c_agr_number', 'acq_class', 'd_valid_from', 'd_valid_to',
            'agr_deleted', 'n_cmp_client', 'cmp_found', 'c_inn', 'cmp_deleted']
ex_agr = pd.DataFrame(columns=agr_cols + ['agr_key', 'Как найден'])
ex_terms = pd.DataFrame(columns=['n_agr', 'cf_ter_type', 'd_valid_from', 'd_valid_to', 'term_deleted'])

if ex_keys:
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        exact = imp.fetch(agr_detail_sql(
            "upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) IN (%s)" % sql_in(ex_keys)))
        exact = exact[exact['ods_key'].isin(ex_keys)].assign(agr_key=lambda d: d['ods_key'], **{'Как найден': 'точно'})

        missing = [k for k in ex_keys if k not in set(exact['agr_key'])]
        digits = {k: re.sub(r'\D', '', k) for k in missing}
        digits = {k: v for k, v in digits.items() if len(v) >= 4}
        fuzzy = pd.DataFrame(columns=agr_cols)
        if digits:
            fuzzy = imp.fetch(agr_detail_sql(
                "regexp_replace(a.c_agr_number, '[^0-9]', '') IN (%s)" % sql_in(set(digits.values()))))
        by_digits = pd.DataFrame({'agr_key': list(digits), 'ods_digits': list(digits.values())}, dtype=object)
        fuzzy['ods_digits'] = fuzzy['ods_digits'].astype(object)
        fuzzy = by_digits.merge(fuzzy, on='ods_digits').assign(**{'Как найден': 'по цифрам номера'})

        ex_agr = pd.concat([exact, fuzzy], ignore_index=True)
        if len(ex_agr):
            ex_terms = imp.fetch(terms_sql(ex_agr['n_agr'].dropna().unique()))

for df in (ex_agr, ex_terms):
    for c in ('d_valid_from', 'd_valid_to'):
        df[c] = pd.to_datetime(df[c], errors='coerce').dt.normalize()
print('Строк договоров:', len(ex_agr), '| строк привязок:', len(ex_terms))


In [ ]:
STEPS = [
    '1. Есть в scd1_agreements',
    '2. Договор не удалён',
    '3. acq_class = SA',
    '4. Действует на %s' % as_of_ru,
    '5. Клиент есть, не удалён, ИНН заполнен',
    '6. Привязка P на %s' % as_of_ru,
]

terms_ok = ex_terms[
    ex_terms['cf_ter_type'].fillna('').str.strip().str.upper().eq('P')
    & ex_terms['term_deleted'].astype(str).ne('1')
    & (ex_terms['d_valid_from'].isna() | (ex_terms['d_valid_from'] <= as_of))
    & (ex_terms['d_valid_to'].isna() | (ex_terms['d_valid_to'] >= as_of))
]
p_on_date = set(terms_ok['n_agr'])
p_any = set(ex_terms.loc[ex_terms['cf_ter_type'].fillna('').str.strip().str.upper().eq('P'), 'n_agr'])

def row_steps(r):
    inn = '' if pd.isna(r['c_inn']) else str(r['c_inn']).strip()
    return [
        True,
        str(r['agr_deleted']) != '1',
        str(r['acq_class'] if pd.notna(r['acq_class']) else '').strip().upper() == 'SA',
        (pd.isna(r['d_valid_from']) or r['d_valid_from'] <= as_of)
        and (pd.isna(r['d_valid_to']) or r['d_valid_to'] >= as_of),
        int(r['cmp_found']) == 1 and str(r['cmp_deleted']) != '1' and inn != '',
        r['n_agr'] in p_on_date,
    ]

def first_fail(steps):
    return next((i for i, ok in enumerate(steps) if not ok), len(steps))

def verdict(fail, r):
    if fail == 1:
        return 'Строка договора помечена удалённой в ods_alpha'
    if fail == 2:
        return 'acq_class = %s, а сборка берёт только SA' % (r['acq_class'] if pd.notna(r['acq_class']) else 'пусто')
    if fail == 3:
        return 'В ods_alpha договор действует с %s по %s, на %s не действует' % (
            fmt(r['d_valid_from']), fmt(r['d_valid_to']) or '...', as_of_ru)
    if fail == 4:
        if int(r['cmp_found']) != 1:
            return 'Клиент n_cmp_client = %s не найден в scd1_companies' % r['n_cmp_client']
        if str(r['cmp_deleted']) == '1':
            return 'Клиент в scd1_companies помечен удалённым'
        return 'У клиента в scd1_companies пустой ИНН'
    if fail == 5:
        if r['n_agr'] in p_any:
            return 'Привязка P есть, но на %s не действует или удалена' % as_of_ru
        return 'Привязок P у договора нет'
    return 'Проходит все шаги, по ods_alpha должен быть в выгрузке: проверить, свежая ли выгрузка'

check_rows = []
for key in ex_keys:
    sn = only_snigur.loc[only_snigur['agr_key'] == key]
    sn = sn.iloc[0] if len(sn) else snigur_agr.loc[snigur_agr['agr_key'] == key].iloc[0]
    base = {
        'Номер договора (Снигур)': sn['Номер договора (Снигур)'],
        'Клиент (Снигур)': sn['Клиент (Снигур)'],
        'Статус (Снигур)': sn['Статус (Снигур)'],
        'Дата начала (Снигур)': sn['Дата начала (Снигур)'],
        'Конец (Снигур)': sn['Конец (Снигур)'],
    }
    rows = ex_agr[ex_agr['agr_key'] == key]
    exact_rows = rows[rows['Как найден'] == 'точно']
    if exact_rows.empty:
        found = rows['c_agr_number'].dropna().astype(str).unique()
        base.update({s: 'нет' if i == 0 else '' for i, s in enumerate(STEPS)})
        base.update({
            'Первый непройденный шаг': STEPS[0],
            'Вывод': ('Номер в ods_alpha записан иначе: %s' % ', '.join(found)) if len(found)
                     else 'Договора с таким номером нет в ods_alpha (и по цифрам номера тоже)',
            'n_agr (ods_alpha)': ', '.join(rows['n_agr'].dropna().astype(str).unique()) or None,
            'n_agr в ods_alpha': rows['n_agr'].nunique(),
        })
        check_rows.append(base)
        continue
    scored = [(first_fail(row_steps(r)), r) for _, r in exact_rows.iterrows()]
    fail, best = max(scored, key=lambda x: x[0])
    steps = row_steps(best)
    base.update({s: ('да' if ok else 'нет') if i <= fail else '' for i, (s, ok) in enumerate(zip(STEPS, steps))})
    base.update({
        'Первый непройденный шаг': STEPS[fail] if fail < len(STEPS) else 'все пройдены',
        'Вывод': verdict(fail, best),
        'n_agr (ods_alpha)': best['n_agr'],
        'n_agr в ods_alpha': exact_rows['n_agr'].nunique(),
    })
    check_rows.append(base)

ex_check = pd.DataFrame(check_rows, columns=[
    'Номер договора (Снигур)', 'Клиент (Снигур)', 'Статус (Снигур)', 'Дата начала (Снигур)', 'Конец (Снигур)',
    *STEPS, 'Первый непройденный шаг', 'Вывод', 'n_agr (ods_alpha)', 'n_agr в ods_alpha',
])

ex_agr_out = ex_agr.rename(columns={
    'agr_key': 'Номер (норм.)', 'n_agr': 'n_agr', 'c_agr_number': 'Номер в ods_alpha', 'acq_class': 'acq_class',
    'd_valid_from': 'Дата начала', 'd_valid_to': 'Дата окончания', 'agr_deleted': 'Договор удалён',
    'n_cmp_client': 'n_cmp_client', 'cmp_found': 'Клиент найден', 'c_inn': 'ИНН', 'cmp_deleted': 'Клиент удалён',
})[['Номер (норм.)', 'Как найден', 'n_agr', 'Номер в ods_alpha', 'acq_class', 'Дата начала', 'Дата окончания',
    'Договор удалён', 'n_cmp_client', 'Клиент найден', 'ИНН', 'Клиент удалён']].sort_values(['Номер (норм.)', 'n_agr'])

agr_by_n = ex_agr[['n_agr', 'c_agr_number']].drop_duplicates('n_agr')
ex_terms_out = ex_terms.merge(agr_by_n, on='n_agr', how='left')
ex_terms_out['Это P'] = ex_terms_out['cf_ter_type'].fillna('').str.strip().str.upper().eq('P').map({True: 'да', False: ''})
ex_terms_out['Действует на %s' % as_of_ru] = (
    (ex_terms_out['d_valid_from'].isna() | (ex_terms_out['d_valid_from'] <= as_of))
    & (ex_terms_out['d_valid_to'].isna() | (ex_terms_out['d_valid_to'] >= as_of))
    & ex_terms_out['term_deleted'].astype(str).ne('1')
).map({True: 'да', False: 'нет'})
ex_terms_out = ex_terms_out.rename(columns={
    'c_agr_number': 'Номер в ods_alpha', 'cf_ter_type': 'cf_ter_type', 'd_valid_from': 'Дата начала',
    'd_valid_to': 'Дата окончания', 'term_deleted': 'Привязка удалена',
})[['Номер в ods_alpha', 'n_agr', 'cf_ter_type', 'Это P', 'Дата начала', 'Дата окончания', 'Привязка удалена',
    'Действует на %s' % as_of_ru]].sort_values(['Номер в ods_alpha', 'n_agr', 'Дата начала'])

display(ex_check)
display(ex_check['Первый непройденный шаг'].value_counts())
display(ex_agr_out)
display(ex_terms_out[ex_terms_out['Это P'] == 'да'])


## 6. Итог для встречи

Четыре таблицы, они же первый лист Excel:
1. сколько договоров и клиентов у Снигура и у меня, сколько есть у обоих и только у одного. Клиент - ИНН, а если его нет - наименование. Клиент «есть у другой стороны», если хотя бы один его договор есть у обоих или тот же клиент найден у другой стороны по ИНН или наименованию (4b);
2. почему у Снигура есть договоры, которых нет у меня;
3. почему у меня есть договоры, которых нет у Снигура;
4. сверка количества: договоры Снигура − только у Снигура + только у меня = мои договоры.


In [ ]:
sn_scope = snigur_active.copy()
sn_scope['inn'] = sn_scope['agr_key'].map(inn_by_key)
sn_scope['name_n'] = sn_scope['Клиент (Снигур)'].map(norm_name)
my_scope = mine.copy()
my_scope['inn'] = my_scope['ИНН'].map(norm_inn)
my_scope['name_n'] = my_scope['agr_key'].map(names_by_key).map(norm_name)
for df in (sn_scope, my_scope):
    df['client'] = df['inn'].where(df['inn'].notna(), 'Н:' + df['name_n'].fillna('?'))

sn_found_client = only_snigur.set_index('agr_key')['Тот же клиент у меня'].str.startswith('Клиент есть')
my_found_client = only_mine.set_index('agr_key')['Тот же клиент у Снигура'].str.startswith('Клиент есть')
sn_scope['matched'] = sn_scope['agr_key'].isin(both_keys) | sn_scope['agr_key'].map(sn_found_client).eq(True)
my_scope['matched'] = my_scope['agr_key'].isin(both_keys) | my_scope['agr_key'].map(my_found_client).eq(True)

def client_counts(df):
    per = df.groupby('client')['matched'].any()
    return len(per), int(per.sum()), int((~per).sum())

sn_cl, my_cl = client_counts(sn_scope), client_counts(my_scope)
meet_counts = pd.DataFrame([
    ('Договоров', len(sn_scope), len(my_scope)),
    ('Клиентов по ИНН', sn_scope['inn'].nunique(), my_scope['inn'].nunique()),
    ('  договоров без ИНН', int(sn_scope['inn'].isna().sum()), int(my_scope['inn'].isna().sum())),
    ('Клиентов по наименованию', sn_scope['name_n'].nunique(), my_scope['name_n'].nunique()),
    ('Клиентов (ИНН, иначе наименование)', sn_cl[0], my_cl[0]),
    ('Договоров есть у обоих', len(both), len(both)),
    ('Договоров только у этой стороны', len(only_snigur), len(only_mine)),
    ('Клиентов есть у другой стороны', sn_cl[1], my_cl[1]),
    ('Клиентов нет у другой стороны', sn_cl[2], my_cl[2]),
], columns=['Показатель', 'Снигур (%s)' % ('весь файл' if SNIGUR_ACTIVE_RULE == 'all' else SNIGUR_ACTIVE_RULE),
            'Моя выгрузка на %s' % as_of_ru])

closed_reason = 'По ods_alpha договор закрыт до %s' % as_of_ru

def sn_meeting_reason(r):
    reason = str(r['Причина'])
    same = str(r.get('Тот же клиент у меня') or '').startswith('Клиент есть')
    if reason == closed_reason:
        st = r['Статус (Снигур)'] if pd.notna(r['Статус (Снигур)']) else 'пусто'
        return 'В Альфе срок закончился до %s, у Снигура статус «%s»' % (as_of_ru, st)
    if 'привязк' in reason.lower():
        return 'В Альфе нет действующей привязки терминала P'
    if reason.startswith('acq_class'):
        return 'В Альфе класс договора не SA'
    if 'начался позже' in reason:
        return 'В Альфе договор начался позже %s' % as_of_ru
    if 'удалён' in reason:
        return 'В Альфе договор удалён'
    if 'ИНН' in reason:
        return 'В Альфе нет клиента или ИНН'
    if reason == 'Возможно, есть у меня под другим номером':
        return 'Похожий номер есть у меня (другая запись номера)'
    if reason == 'Договора нет в ods_alpha':
        return ('Номера нет в Альфе, тот же клиент есть у меня под другим номером' if same
                else 'Номера нет в Альфе, клиента у меня нет')
    if reason.startswith('Проходит все фильтры'):
        return 'Проходит все фильтры моей сборки, проверить вручную'
    return reason

def my_meeting_reason(r):
    same = str(r.get('Тот же клиент у Снигура') or '')
    if pd.notna(r.get('Пара по цифрам номера')):
        return 'Похожий номер есть у Снигура (другая запись номера)'
    if str(r['Причина']).startswith('У Снигура не действует'):
        return 'Договор есть у Снигура, но не действует на %s' % as_of_ru
    if 'другой договор есть у обоих' in same:
        return 'Другой договор этого клиента есть у обоих'
    if 'под другим номером' in same:
        return 'Тот же клиент у Снигура под другим номером'
    if r.get('Как найден (нечётко)') in ('ИП по фамилии и инициалам', 'по словам', 'похожесть строк'):
        return 'Клиент найден у Снигура только нечётко по наименованию'
    return 'Клиента у Снигура нет ни по ИНН, ни по наименованию'

def reason_table(df, fn, title):
    s = (df.apply(fn, axis=1) if len(df) else pd.Series(dtype=object)).value_counts()
    t = s.rename_axis(title).reset_index(name='Договоров')
    t['Доля, %'] = (t['Договоров'] / max(len(df), 1) * 100).round(1)
    total = pd.DataFrame([{title: 'Всего', 'Договоров': len(df), 'Доля, %': 100.0}])
    return pd.concat([t, total], ignore_index=True)

meet_sn = reason_table(only_snigur, sn_meeting_reason, 'Только у Снигура: почему нет у меня')
meet_mine = reason_table(only_mine, my_meeting_reason, 'Только у меня: почему нет у Снигура')

calc = len(sn_scope) - len(only_snigur) + len(only_mine)
meet_math = pd.DataFrame([
    ('Договоров у Снигура', len(sn_scope)),
    ('− только у Снигура', len(only_snigur)),
    ('+ только у меня', len(only_mine)),
    ('= договоров у меня (расчёт)', calc),
    ('Договоров у меня (факт)', len(my_scope)),
    ('Сходится', 'да' if calc == len(my_scope) else 'нет'),
], columns=['Сверка количества', 'Значение'])

for t in (meet_counts, meet_sn, meet_mine, meet_math):
    display(t)


## 7. Выгрузка для Ани

In [ ]:
sn_cols = ['Клиент (Снигур)', 'Номер договора (Снигур)', 'Дата начала (Снигур)', 'Дата окончания (Снигур)',
           'Дата закрытия (Снигур)', 'Статус (Снигур)', 'Противоречие статуса', 'Строк у Снигура']
mine_cols = ['ИНН', 'ЦФТ ID', 'Номер договора (мой)', 'Дата открытия (мой)', 'Дата закрытия (мой)', 'РФ', 'Строк у меня']
ods_cols = ['n_agr (ods_alpha)', 'n_agr в ods_alpha', 'acq_class (ods_alpha)', 'Дата начала (ods_alpha)',
            'Дата окончания (ods_alpha)', 'ИНН (ods_alpha)', 'Привязка P на дату']

sheet_both = both[sn_cols + mine_cols + ['Даты расходятся', 'Разница начала, дней', 'Разница окончания, дней']].sort_values('Номер договора (Снигур)')
sheet_only_sn = only_snigur[
    ['Причина', 'Пара по цифрам номера', 'Тот же клиент у меня', 'Договоры клиента у меня', 'Мой договор новее']
    + sn_cols + ods_cols
].sort_values(['Причина', 'Номер договора (Снигур)'])
sheet_only_mine = only_mine[
    ['Причина', 'Пара по цифрам номера', 'Договоры клиента у Снигура', 'У Снигура договор новее моего', 'Наименование (ods_alpha)']
    + mine_cols + sn_cols + ods_cols[2:]
].sort_values(['Причина', 'Номер договора (мой)'])

summary = pd.DataFrame([
    ('Дата сверки', as_of_ru),
    ('Правило «действует у Снигура»', SNIGUR_ACTIVE_RULE),
    ('Строк в файле Снигура', len(snigur)),
    ('Договоров у Снигура (уникальных номеров)', len(snigur_agr)),
    ('Договоров у Снигура, действующих на дату', len(snigur_active)),
    ('Договоров в моей выгрузке', len(mine)),
    ('Есть у обоих', len(sheet_both)),
    ('  из них даты расходятся', int(sheet_both['Даты расходятся'].notna().sum())),
    ('Только у Снигура', len(sheet_only_sn)),
    ('Только у меня', len(sheet_only_mine)),
    ('Возможно один договор с разной записью номера (пар)', len(num_pairs)),
    ('У Снигура «Дата окончания» раньше даты сверки, но по дате закрытия действует', int(sn_end_passed.sum())),
], columns=['Показатель', 'Значение'])

reasons_sn = sheet_only_sn['Причина'].value_counts().rename_axis('Только у Снигура: причина').reset_index(name='Договоров')
reasons_mine = (
    sheet_only_mine['Причина'].str.replace(r':.*', '', regex=True)
    .value_counts().rename_axis('Только у меня: причина').reset_index(name='Договоров')
)
statuses = (
    pd.crosstab(snigur_agr['Статус (Снигур)'].fillna('(пусто)'), snigur_agr['Действует на дату'].map({True: 'Действует', False: 'Не действует'}))
    .rename_axis('Статус у Снигура').reset_index()
)

def diff_counts(mine_keys):
    sn_keys = set(snigur_active['agr_key'])
    return [len(mine_keys), len(mine_keys & sn_keys), len(sn_keys - mine_keys), len(mine_keys - sn_keys)]

now_keys = set(mine['agr_key'])
if BEFORE_FILE.exists():
    before_raw = pd.read_excel(BEFORE_FILE, sheet_name=MINE_SHEET, dtype=object)
    before_keys = set(before_raw['Номер договора'].map(norm_agr).dropna())
    before_after = pd.DataFrame({
        'До и после правок озера': ['Договоров в моей выгрузке', 'Есть у обоих', 'Только у Снигура', 'Только у меня',
                                    'Добавились в моей выгрузке', 'Пропали из моей выгрузки'],
        'Было': pd.array(diff_counts(before_keys) + [None, None], dtype='Int64'),
        'Стало': pd.array(diff_counts(now_keys) + [len(now_keys - before_keys), len(before_keys - now_keys)], dtype='Int64'),
    })
else:
    before_after = pd.DataFrame({'До и после правок озера': ['Нет файла %s' % BEFORE_FILE.name]})
display(before_after)

ex_steps = (
    ex_check['Первый непройденный шаг'].value_counts()
    .rename_axis('Примеры: первый непройденный шаг').reset_index(name='Договоров')
)

def as_dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

def autofit(ws):
    for col in ws.columns:
        width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 10), 60)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:
    row = 0
    for block in (meet_counts, meet_math, meet_sn, meet_mine):
        block.to_excel(writer, index=False, sheet_name='Итог для встречи', startrow=row)
        row += len(block) + 2
    row = 0
    for block in (summary, before_after, per_client, same_client_mine, newer_mine, same_client_sn,
                  fuzzy_sum, fuzzy_status, nosn_cft, nosn_rf, nosn_year, nosn_month, reasons_sn, closed_years, closed_months_2026, end_vs_ods, reason_by_status,
                  reasons_mine, date_kinds, start_diff_top,
                  end_patterns, statuses, closed_by_month, close_by_status, ex_steps):
        block.to_excel(writer, index=False, sheet_name='Сводка', startrow=row)
        row += len(block) + 2
    as_dates(sheet_both).to_excel(writer, index=False, sheet_name='Есть у обоих')
    as_dates(sheet_only_sn).to_excel(writer, index=False, sheet_name='Только у Снигура')
    as_dates(sheet_only_mine).to_excel(writer, index=False, sheet_name='Только у меня')
    num_pairs.to_excel(writer, index=False, sheet_name='Разный формат номера')
    as_dates(fuzzy_out).to_excel(writer, index=False, sheet_name='Нечёткий поиск')
    manual_out.to_excel(writer, index=False, sheet_name='10 примеров - вручную')
    as_dates(ex_check).to_excel(writer, index=False, sheet_name='Примеры - проверка')
    as_dates(ex_agr_out).to_excel(writer, index=False, sheet_name='Примеры - договоры')
    as_dates(ex_terms_out).to_excel(writer, index=False, sheet_name='Примеры - привязки P')
    for ws in writer.book.worksheets:
        autofit(ws)
        ws.freeze_panes = 'A2' if ws.title not in ('Сводка', 'Итог для встречи') else None

display(summary)
print(OUT_FILE)
